In [3]:
import os
import shutil

base_dir = 'train'
cats_dir = os.path.join(base_dir, 'cats')
dogs_dir = os.path.join(base_dir, 'dogs')

os.makedirs(cats_dir, exist_ok=True)
os.makedirs(dogs_dir, exist_ok=True)

for filename in os.listdir(base_dir):
    if filename.lower().startswith('cat'):
        shutil.move(os.path.join(base_dir, filename), cats_dir)
    elif filename.lower().startswith('dog'):
        shutil.move(os.path.join(base_dir, filename), dogs_dir)

print("✅ تم نقل الصور لمجلدات cats و dogs بنجاح!")


✅ تم نقل الصور لمجلدات cats و dogs بنجاح!


In [1]:
# =======================================================
# 🐾 Cat vs Dog Classifier using CNN
#  Rashid osama fathi shehadeh 
# =======================================================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image
import os


train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)


train_set = train_datagen.flow_from_directory(
    'train',
    target_size=(64, 64),
    batch_size=16,
    class_mode='binary'
)


model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_set, epochs=3)


model.save('cat_dog_cnn_model.h5')
print("✅ Model saved as cat_dog_cnn_model.h5")


test_folder = 'test'
test_images = sorted(os.listdir(test_folder))
predictions = []

for img_name in test_images:
    img_path = os.path.join(test_folder, img_name)
    img = image.load_img(img_path, target_size=(64,64))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    result = model.predict(img_array, verbose=0)
    predictions.append(int(result[0][0] > 0.5))  # 0=cat, 1=dog


submission = pd.DataFrame({
    'id': [int(os.path.splitext(img_name)[0]) for img_name in test_images],
    'label': predictions
})
submission.to_csv('submission.csv', index=False)

print("✅ تم إنشاء الملف submission.csv بنجاح!")
print("🎯 جاهز للرفع على Kaggle")


Found 25000 images belonging to 2 classes.


C:\Users\rashi\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\rashi\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 429s 274ms/step - accuracy: 0.6049 - loss: 0.6543
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 183s 117ms/step - accuracy: 0.7386 - loss: 0.5333
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 136s 87ms/step - accuracy: 0.7639 - loss: 0.4900


✅ Model saved as cat_dog_cnn_model.h5
✅ تم إنشاء الملف submission.csv بنجاح!
🎯 جاهز للرفع على Kaggle


In [5]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

model = load_model('cat_dog_cnn_model.h5')

img_path = 'test/12022.jpg' 
img = image.load_img(img_path, target_size=(64,64))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

result = model.predict(img_array)
if result[0][0] > 0.5:
    print("🐶 It's a Dog!")
else:
    print("🐱 It's a Cat!")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
🐶 It's a Dog!
